In [ ]:
%pip install soundfile numpy
import soundfile as sf

# Read the audio file
# Output is Snapshot: At one specific micro-moment, the computer records the air pressure (the amplitude) of the sound wave. This single measurement is a sample.
#The Sample Rate: If the sample rate is 44,100 Hz, it means the computer took 44,100 snapshots in just one second.

data, samplerate = sf.read('./ElevenLabs_2025-11-22T03_21_45_Rachel_ Subject-Verb Agreement.mp3')

# 'data' is a NumPy array, 'samplerate' is the rate in Hz
print(f"Sampling Rate: {samplerate}")
print(f"Audio Data: {data}")

  Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl.metadata (16 kB)
Using cached soundfile-0.13.1-py2.py3-none-win_amd64.whl (1.0 MB)
Note: you may need to restart the kernel to use updated packages.
Sampling Rate: 44100
Audio Data: [ 0.00000000e+00 -1.02720627e-11 -8.50996357e-12 ...  0.00000000e+00
  0.00000000e+00  0.00000000e+00]


In [1]:
%pip install SpeechRecognition
# You may need pydub if using MP3 files
%pip install pydub
# You may need an FFmpeg installation for pydub to work

   ---------------------------------------- 0.0/32.9 MB ? eta -:--:--
   -- ------------------------------------- 2.1/32.9 MB 13.0 MB/s eta 0:00:03
   ------ --------------------------------- 5.0/32.9 MB 13.7 MB/s eta 0:00:03
   --------- ------------------------------ 7.6/32.9 MB 13.0 MB/s eta 0:00:02
   ------------ --------------------------- 10.2/32.9 MB 13.0 MB/s eta 0:00:02
   --------------- ------------------------ 12.6/32.9 MB 12.7 MB/s eta 0:00:02
   ------------------ --------------------- 15.2/32.9 MB 12.9 MB/s eta 0:00:02
   --------------------- ------------------ 17.8/32.9 MB 12.8 MB/s eta 0:00:02
   ------------------------- -------------- 20.7/32.9 MB 12.8 MB/s eta 0:00:01
   ---------------------------- ----------- 23.3/32.9 MB 12.8 MB/s eta 0:00:01
   ------------------------------- -------- 26.0/32.9 MB 12.9 MB/s eta 0:00:01
   ----------------------------------- ---- 28.8/32.9 MB 12.9 MB/s eta 0:00:01
   ------------------------------------- -- 31.2/32.9 MB 12.9 MB

In [7]:
from langsmith import traceable
from openai import OpenAI

from openai import OpenAI
import os
from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env.local")
my_openai_client = os.getenv("OPENAI_API_KEY")

# Initialize client (make sure OPENAI_API_KEY is set in env vars)
client = OpenAI()

@traceable # Tracks the transcription step
def transcribe_audio(audio_file_path: str) -> str:
    """
    Reads an audio file and returns its transcript using GPT transcription model.
    """
    with open(audio_file_path, "rb") as audio_file:
        transcript = client.audio.transcriptions.create(
            file=audio_file,
            model="gpt-4o-mini-transcribe"
        )

    return transcript.text


if __name__ == "__main__":
    audio_path = "ElevenLabs_2025-11-22T03_21_45_Rachel_ Subject-Verb Agreement.mp3" 

    text = transcribe_audio(audio_path)
    print("\n--- TRANSCRIPT ---\n")
    print(text)



--- TRANSCRIPT ---

Every morning he go to school with his friends. They walks together down the street and talks about their homework. The teacher always say that punctuality are important. My sister also go to the same school and she love her classes. The students in the classroom enjoys learning new things every day.


In [8]:
import json

SYSTEM_PROMPT_SPEECH_EVAL = """
You are a senior communication coach, linguistics expert, and AI evaluation system.

Your task is to analyze spoken or written transcripts and evaluate communication quality across four dimensions:

1. Grammar accuracy (25%)
2. Clarity and coherence (25%)
3. Vocabulary richness (25%)
4. Sentence structure (25%)

Follow this process internally:
- Think through the evaluation step-by-step.
- Carefully assess grammar, clarity, vocabulary, and sentence patterns.
- Assign calibrated scores based on professional communication standards.
- Generate concise, actionable coaching feedback.

IMPORTANT:
- Do NOT show your reasoning.
- Output ONLY the final JSON result.

Few-shot examples:

Example 1:
Transcript:
"I think the meeting went pretty good and we talked about stuff and it was like fine."

Output:
{
  "grammar_accuracy": {"score": 82, "feedback": "Usaing "good" (adjective) instead of the adverb "well" to modify the verb "went is a grammatical error." The use of "like" as filler is also informal. "},
  "clarity_and_coherence": {"score": 74, "feedback": "The core message is understandable, but "stuff" is vague and "pretty good" and "fine" are non-descriptive, requiring the listener to infer the actual outcomes. Ideas are understandable but lack structured flow and precision."},
  "vocabulary_richness": {"score": 68, "feedback": "The vocabulary is highly informal and basic ("pretty good," "talked about stuff," "like fine"), lacks precise terminology often used in professional contexts to describe meeting outcomes."},
  "sentence_structure": {"score": 72, "feedback": "The single compound sentence uses repetitive coordinating conjunctions ("and")."},
  "overall_summary": "Clear but informal delivery; improving structure and word choice would significantly elevate professionalism. Minor grammatical corrections and tense consistency would improve fluency."
}

Example 2:
Transcript:
"Our quarterly performance exceeded expectations, driven by stronger client engagement and operational efficiency."

Output:
{
  "grammar_accuracy": {"score": 96, "feedback": "Grammar is strong with professional-level accuracy."},
  "clarity_and_coherence": {"score": 94, "feedback": "Ideas are clearly expressed and logically organized."},
  "vocabulary_richness": {"score": 92, "feedback": "Vocabulary is precise, varied, and context-appropriate."},
  "sentence_structure": {"score": 91, "feedback": "Sentence structure is strong with effective complexity and flow."},
  "overall_summary": "Very professional, clear, and effective communication."
}

Now evaluate the following transcript.

Rules:
- Provide a numeric score from 0 to 100 for each category.
- Provide concise, precise feedback in 1–2 lines.
- Focus on actionable improvement suggestions.
- Output strictly in valid JSON using this format:

{
  "grammar_accuracy": {"score": number, "feedback": "string"},
  "clarity_and_coherence": {"score": number, "feedback": "string"},
  "vocabulary_richness": {"score": number, "feedback": "string"},
  "sentence_structure": {"score": number, "feedback": "string"},
  "overall_summary": "1–2 line summary"
}
"""


In [9]:
@traceable(run_type="llm") # Categorizes this specific span as an LLM call
def evaluate_transcript(transcript: str) -> dict:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=[
            {"role": "system", "content": SYSTEMSYSTEM_PROMPT_SPEECH_EVAL_PROMPT},
            {"role": "user", "content": transcript}
        ]
    )

    return json.loads(response.choices[0].message.content)


if __name__ == "__main__":
    transcript = """
    I think the meeting went pretty well. We talked about the project goals and
    everyone seemed aligned but maybe we can improve communication next time.
    """
    transcript = transcribe_audio("ElevenLabs_2025-11-22T03_21_45_Rachel_ Subject-Verb Agreement.mp3")
    feedback = evaluate_transcript(transcript)

    print("\n--- AI FEEDBACK ---\n")
    print(json.dumps(feedback, indent=2))


NameError: name 'SYSTEMSYSTEM_PROMPT_SPEECH_EVAL_PROMPT' is not defined

In [22]:

SYSTEM_PROMPT_FILLER_WORDS = """
You are a professional language editor.

Task:
Identify filler words and grammatical errors in the text.

Rules:
- Wrap ONLY filler words and grammatical errors in *asterisks*.
- Do NOT rewrite the text.
- Do NOT add or remove words.
- Preserve original word order, punctuation, and casing.
- Only highlight problematic words or phrases.

Common filler words include:
um, uh, like, you know, kind of, sort of, basically, actually, literally, so, well.

Return only the final corrected text.
"""




In [23]:
def highlight_filler_words_text(text: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.1,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT_FILLER_WORDS},
            {"role": "user", "content": text}
        ]
    )

    return response.choices[0].message.content.strip()


if __name__ == "__main__":
    text = """
    Um I was like thinking that we kind of should go there yesterday but it don't make sense.
    """

    result = highlight_filler_words_text(text)

    print("\n--- HIGHLIGHTED TEXT ---\n")
    print(result)


--- HIGHLIGHTED TEXT ---

*Um* I was *like* thinking that we *kind of* should go there *yesterday* but it *don't* make sense.
